# 01 — M1 Corpus Builder V3

**Goal:** Build a balanced binary corpus (safe / harmful) for the M1 safety gate.

**V3 Changes vs V2:**
1. `normalize_text()` applied to ALL text before anything else
2. Safe pool expanded from 4 sources → 11 sources (hard negatives from same register as harmful)
3. TRAC-3 files removed (not English)
4. Target: **100K rows (50K safe / 50K harmful)**

**Execution order:** Run cells 1 → 2 → 3 → 4 → 5 → 6 sequentially.

In [12]:
# ============================================================
# CELL 1 — Setup + Drive Mount + Path Inventory
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import hashlib
from collections import Counter

# --- Paths ---
BASE      = Path("/content/drive/MyDrive/AEGIS_ML")
RAW       = BASE / "data" / "raw"
THREAT    = RAW / "threat_gap"
PROCESSED = BASE / "data" / "processed" / "v5"
PROCESSED.mkdir(parents=True, exist_ok=True)

# --- Verify key files ---
print("="*70)
print("CELL 1: SETUP + PATH INVENTORY")
print("="*70)

key_files = {
    'jigsaw':              RAW / 'jigsaw' / 'train.csv',
    'davidson':            RAW / 'davidson.csv',
    'hatexplain':          RAW / 'hatexplain.json',
    'edos':                RAW / 'edos.csv',
    'cash':                RAW / 'cash_sexual_harassment_en.csv',
    'tweeteval_offensive': RAW / 'tweeteval_offensive.csv',
    'tweeteval_hate':      RAW / 'tweeteval_hate.csv',
    'mlma':                RAW / 'mlma.csv',
    'kaggle_cb':           RAW / 'cyberbullying' / 'cyberbullying_tweets.csv',
    'stormfront':          RAW / 'stormfront_hate_speech18.csv',
    'toxigen':             RAW / 'toxigen_adversarial.csv',
    'aegis_safety':        RAW / 'aegis_safety_corpus.csv',
    'olid':                RAW / 'olid' / 'olid_full.tsv',
    'gab_routed':          THREAT / 'gab_routed.csv',
    'hammer':              THREAT / 'hammer_threat.csv',
    'ethos':               THREAT / 'ethos_threat.csv',
    'implicit_hate':       THREAT / 'implicit_hate_threat.csv',
    'ucb_mhs_threat':      THREAT / 'ucb_mhs_threat.csv',
    'ucb_mhs_disc':        THREAT / 'ucb_mhs_discrimination.csv',
    'convabuse_threat':    THREAT / 'convabuse_threat.csv',
    'convabuse_sexual':    THREAT / 'convabuse_sexual_harassment.csv',
    'convabuse_verbal':    THREAT / 'convabuse_verbal_harassment.csv',
    'convabuse_disc':      THREAT / 'convabuse_discrimination.csv',
    'hasoc_2019_train':    RAW / 'hosoc_english' / 'english_dataset_2019' / 'english_dataset.tsv',
    'hasoc_2019_test':     RAW / 'hosoc_english' / 'english_dataset_2019' / 'hasoc2019_en_test-2919.tsv',
    'hasoc_2020_train':    RAW / 'hosoc_english' / 'english_dataset_2020' / 'hasoc_2020_en_train_new.xlsx',
    'hasoc_2020_test':     RAW / 'hosoc_english' / 'english_dataset_2020' / 'hasoc_2020_en_test_new.xlsx',
}

print("\n📁 Checking key files:")
for name, path in key_files.items():
    exists = '✅' if path.exists() else '⚠️  MISSING'
    size = f"{path.stat().st_size/1024:.1f} KB" if path.exists() else ''
    print(f"   {exists} {name:<25} {size}")

print(f"\n{'='*70}")
print("✅ CELL 1 COMPLETE — Ready for Cell 2")
print(f"{'='*70}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CELL 1: SETUP + PATH INVENTORY

📁 Checking key files:
   ✅ jigsaw                    67190.1 KB
   ✅ davidson                  2352.1 KB
   ✅ hatexplain                11968.9 KB
   ✅ edos                      3756.8 KB
   ✅ cash                      4228.6 KB
   ✅ tweeteval_offensive       1510.8 KB
   ✅ tweeteval_hate            1122.4 KB
   ✅ mlma                      2682.3 KB
   ✅ kaggle_cb                 7006.4 KB
   ✅ stormfront                1315.5 KB
   ✅ toxigen                   3313.0 KB
   ✅ aegis_safety              9475.0 KB
   ⚠️  MISSING olid                      
   ✅ gab_routed                4629.8 KB
   ✅ hammer                    161.3 KB
   ✅ ethos                     22.4 KB
   ✅ implicit_hate             245.0 KB
   ✅ ucb_mhs_threat            235.5 KB
   ✅ ucb_mhs_disc              212.7 KB
   ✅ convabuse_threat          4.5 KB
   

In [13]:
# ============================================================
# CELL 2 — Define normalize_text() + Helpers
# ============================================================

import re
import hashlib

def normalize_text(text: str) -> str:
    """Canonical text normalization for AEGIS pipeline.
    Applied to ALL text BEFORE tokenization.
    Eliminates casing/punctuation sensitivity."""
    text = str(text).strip()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)      # URLs
    text = re.sub(r'@\w+', '', text)                         # @mentions → remove
    text = re.sub(r'#(\w+)', r'\1', text)                    # #hashtags → hashtags
    text = re.sub(r'\brt\s+', '', text)                      # RT prefix
    text = re.sub(r'&amp;', '&', text)                       # HTML entities
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&#\d+;', '', text)                       # numeric entities
    text = re.sub(r'<user>', '', text)                        # HateXplain tokens
    text = re.sub(r'\s+', ' ', text).strip()                 # collapse whitespace
    return text if text else None

def make_frame(texts, labels, source):
    """Create standardized frame with [text, m1_label, source_dataset]."""
    df = pd.DataFrame({'text': texts, 'm1_label': labels, 'source_dataset': source})
    df = df[df['text'].notna() & df['m1_label'].notna()]
    # Apply normalization immediately
    df['text'] = df['text'].apply(normalize_text)
    df = df[df['text'].notna()]
    df = df[df['text'].str.len() > 0]
    print(f"  ✅ {source:<35} {len(df):>7,} rows  |  {df['m1_label'].value_counts().to_dict()}")
    return df

def text_hash(t):
    """SHA-256 hash for exact dedup."""
    return hashlib.sha256(t.strip().encode('utf-8')).hexdigest()

print("✅ normalize_text() defined")
print("✅ make_frame() defined (includes normalization)")
print("✅ text_hash() defined")

# Quick test
assert normalize_text("RT @user Check THIS out!! http://t.co/abc") == "check this out!!"
assert normalize_text("SEND me your NUDES") == "send me your nudes"
assert normalize_text("I like this game") == "i like this game"
print("\n✅ All normalization tests passed")

✅ normalize_text() defined
✅ make_frame() defined (includes normalization)
✅ text_hash() defined

✅ All normalization tests passed


In [14]:
# ============================================================
# CELL 3 — Load & Route ALL Datasets → M1 Binary Labels
# Output: one df per source with columns [text, m1_label, source_dataset]
# m1_label: 'safe' or 'harmful'
# ALL text is normalized via make_frame()
# ============================================================
import pandas as pd
import numpy as np
import json
from collections import Counter

frames = []  # collect all routed dfs

print("="*70)
print("CELL 3: LOAD & ROUTE ALL DATASETS")
print("="*70)

# ── 1. JIGSAW ────────────────────────────────────────────────
print("\n[1] Jigsaw")
df_j = pd.read_csv(RAW / "jigsaw" / "train.csv")
harmful_mask = (df_j[['toxic','severe_toxic','obscene','threat','insult','identity_hate']].sum(axis=1) > 0)
frames.append(make_frame(
    df_j['comment_text'],
    harmful_mask.map({True:'harmful', False:'safe'}),
    'jigsaw'
))

# ── 2. DAVIDSON ───────────────────────────────────────────────
print("\n[2] Davidson")
df_d = pd.read_csv(RAW / "davidson.csv")
label_map = {0:'harmful', 1:'harmful', 2:'safe'}
frames.append(make_frame(
    df_d['tweet'],
    df_d['class'].map(label_map),
    'davidson'
))

# ── 3. HATEXPLAIN ─────────────────────────────────────────────
print("\n[3] HateXplain")
with open(RAW / "hatexplain.json") as f:
    hx_raw = json.load(f)

hx_rows = []
for entry in hx_raw.values():
    text = ' '.join(entry['post_tokens'])
    votes = [a['label'] for a in entry['annotators']]
    majority = Counter(votes).most_common(1)[0][0]
    m1 = 'safe' if majority == 'normal' else 'harmful'
    hx_rows.append({'text': text, 'm1_label': m1, 'source_dataset': 'hatexplain'})

df_hx = pd.DataFrame(hx_rows)
# Apply normalization
df_hx['text'] = df_hx['text'].apply(normalize_text)
df_hx = df_hx[df_hx['text'].notna() & (df_hx['text'].str.len() > 0)]
print(f"  ✅ {'hatexplain':<35} {len(df_hx):>7,} rows  |  {df_hx['m1_label'].value_counts().to_dict()}")
frames.append(df_hx)

# ── 4. EDOS ───────────────────────────────────────────────────
print("\n[4] EDOS")
df_e = pd.read_csv(RAW / "edos.csv")
frames.append(make_frame(
    df_e['text'],
    df_e['label_sexist'].map({'not sexist':'safe', 'sexist':'harmful'}),
    'edos'
))

# ── 5. CASH ───────────────────────────────────────────────────
print("\n[5] CASH")
df_cash = pd.read_csv(RAW / "cash_sexual_harassment_en.csv")
frames.append(make_frame(
    df_cash['text'],
    'harmful',
    'cash'
))

# ── 6. TWEETEVAL OFFENSIVE ───────────────────────────────────
print("\n[6] TweetEval Offensive")
df_to = pd.read_csv(RAW / "tweeteval_offensive.csv")
frames.append(make_frame(
    df_to['text'],
    df_to['label'].map({0:'safe', 1:'harmful'}),
    'tweeteval_offensive'
))

# ── 7. TWEETEVAL HATE ────────────────────────────────────────
print("\n[7] TweetEval Hate")
df_th = pd.read_csv(RAW / "tweeteval_hate.csv")
frames.append(make_frame(
    df_th['text'],
    df_th['label'].map({0:'safe', 1:'harmful'}),
    'tweeteval_hate'
))

# ── 8. MLMA ───────────────────────────────────────────────────
print("\n[8] MLMA")
df_mlma = pd.read_csv(RAW / "mlma.csv")
frames.append(make_frame(
    df_mlma['tweet'],
    df_mlma['sentiment'].map(lambda x: 'safe' if str(x).strip().lower()=='normal' else 'harmful'),
    'mlma'
))

# ── 9. UCB MHS ────────────────────────────────────────────────
print("\n[9] UCB MHS (pre-filtered)")
df_ucbt = pd.read_csv(THREAT / "ucb_mhs_threat.csv")
df_ucbd = pd.read_csv(THREAT / "ucb_mhs_discrimination.csv")
df_ucb  = pd.concat([df_ucbt, df_ucbd], ignore_index=True)
frames.append(make_frame(df_ucb['text'], 'harmful', 'ucb_mhs'))

# ── 10. KAGGLE CYBERBULLYING ──────────────────────────────────
print("\n[10] Kaggle Cyberbullying")
df_cb = pd.read_csv(RAW / "cyberbullying" / "cyberbullying_tweets.csv")
frames.append(make_frame(
    df_cb['tweet_text'],
    df_cb['cyberbullying_type'].map(
        lambda x: 'safe' if x == 'not_cyberbullying' else 'harmful'
    ),
    'kaggle_cyberbullying'
))

# ── 11. GAB (already routed) ─────────────────────────────────
print("\n[11] Gab (routed)")
df_gab = pd.read_csv(THREAT / "gab_routed.csv")
frames.append(make_frame(
    df_gab['text'],
    df_gab['label'].map(lambda x: 'safe' if x == 'safe' else 'harmful'),
    'gab'
))

# ── 12. OLID (already routed) ────────────────────────────────
print("\n[12] OLID (routed)")
df_olid = pd.read_csv(THREAT / "olid_routed.csv")
frames.append(make_frame(
    df_olid['text'],
    df_olid['label'].map(lambda x: 'safe' if x == 'safe' else 'harmful'),
    'olid'
))

# ── 13. AEGIS SAFETY CORPUS ───────────────────────────────────
print("\n[13] AEGIS Safety Corpus (user_message only)")
df_asc = pd.read_csv(RAW / "aegis_safety_corpus.csv")
df_asc = df_asc[df_asc['text_type'] == 'user_message'].copy()
label_cols = ['labels_0','labels_1','labels_2','labels_3','labels_4']
def aegis_vote(row):
    votes = [str(row[c]).strip().lower() for c in label_cols if pd.notna(row[c])]
    if not votes: return None
    safe_votes = sum(1 for v in votes if v == 'safe')
    return 'safe' if safe_votes > len(votes)/2 else 'harmful'
df_asc['m1_label'] = df_asc.apply(aegis_vote, axis=1)
frames.append(make_frame(df_asc['text'], df_asc['m1_label'], 'aegis_safety'))

# ── 14. IMPLICIT HATE ────────────────────────────────────────
print("\n[14] ImplicitHate threat")
df_ih = pd.read_csv(THREAT / "implicit_hate_threat.csv")
frames.append(make_frame(df_ih['text'], 'harmful', 'implicit_hate'))

# ── 15. HAMMER ────────────────────────────────────────────────
print("\n[15] Hammer threat")
df_hm = pd.read_csv(THREAT / "hammer_threat.csv")
frames.append(make_frame(df_hm['text'], 'harmful', 'hammer'))

# ── 16. ETHOS ────────────────────────────────────────────────
print("\n[16] ETHOS threat")
df_et = pd.read_csv(THREAT / "ethos_threat.csv")
frames.append(make_frame(df_et['text'], 'harmful', 'ethos'))

# ── 17. CONVABUSE (all 4 pre-filtered files) ─────────────────
print("\n[17] ConvAbuse")
ca_files = ['convabuse_threat.csv','convabuse_sexual_harassment.csv',
            'convabuse_verbal_harassment.csv','convabuse_discrimination.csv']
df_ca = pd.concat([pd.read_csv(THREAT / f) for f in ca_files], ignore_index=True)
frames.append(make_frame(df_ca['text'], 'harmful', 'convabuse'))

# ── 18. STORMFRONT ───────────────────────────────────────────
print("\n[18] Stormfront")
df_sf = pd.read_csv(RAW / "stormfront_hate_speech18.csv")
sf_map = {'hate':'harmful', 'noHate':'safe'}
df_sf['m1_label'] = df_sf['label'].map(sf_map)
frames.append(make_frame(df_sf['text'], df_sf['m1_label'], 'stormfront'))

# ── 19. TOXIGEN ───────────────────────────────────────────────
print("\n[19] ToxiGen")
df_tg = pd.read_csv(RAW / "toxigen_adversarial.csv")
score_col = 'toxicity_human' if 'toxicity_human' in df_tg.columns else 'toxicity_ai'
df_tg['m1_label'] = df_tg[score_col].apply(
    lambda x: 'harmful' if pd.notna(x) and float(x) >= 0.5 else 'safe'
)
frames.append(make_frame(df_tg['text'], df_tg['m1_label'], 'toxigen'))

# ── 20. HASOC (NEW — English 2019 + 2020) ────────────────────
print("\n[20] HASOC English (2019 + 2020)")
try:
    h19_train = pd.read_csv(RAW / 'hosoc_english/english_dataset_2019/english_dataset.tsv', sep='\t')
    h19_test  = pd.read_csv(RAW / 'hosoc_english/english_dataset_2019/hasoc2019_en_test-2919.tsv', sep='\t')
    h19 = pd.concat([h19_train, h19_test], ignore_index=True)
    h19 = h19.rename(columns={'task_1': 'task1'})

    h20_train = pd.read_excel(RAW / 'hosoc_english/english_dataset_2020/hasoc_2020_en_train_new.xlsx')
    h20_test  = pd.read_excel(RAW / 'hosoc_english/english_dataset_2020/hasoc_2020_en_test_new.xlsx')
    h20 = pd.concat([h20_train, h20_test], ignore_index=True)

    hasoc = pd.concat([h19[['text','task1']], h20[['text','task1']]], ignore_index=True)
    hasoc['m1_label'] = hasoc['task1'].map({'NOT': 'safe', 'HOF': 'harmful'})
    frames.append(make_frame(hasoc['text'], hasoc['m1_label'], 'hasoc'))
except Exception as e:
    print(f"  ⚠️  HASOC loading failed: {e}")

# ── COMBINE ALL ───────────────────────────────────────────────
print("\n" + "="*70)
print("COMBINED — RAW (before dedup + caps)")
print("="*70)
df_all = pd.concat(frames, ignore_index=True)
df_all = df_all[['text','m1_label','source_dataset']]

print(f"\nTotal rows      : {len(df_all):,}")
print(f"\nm1_label counts :\n{df_all['m1_label'].value_counts()}")
print(f"\nPer source      :")
print(df_all.groupby(['source_dataset','m1_label']).size().unstack(fill_value=0).to_string())

print(f"\n{'='*70}")
print("✅ CELL 3 COMPLETE — All sources loaded + normalized")
print("   Next: Cell 4 (PersonaChat + DailyDialog + synthetic hard negatives)")
print(f"{'='*70}")

CELL 3: LOAD & ROUTE ALL DATASETS

[1] Jigsaw
  ✅ jigsaw                              159,564 rows  |  {'safe': 143339, 'harmful': 16225}

[2] Davidson
  ✅ davidson                             24,783 rows  |  {'harmful': 20620, 'safe': 4163}

[3] HateXplain
  ✅ hatexplain                           20,148 rows  |  {'harmful': 11995, 'safe': 8153}

[4] EDOS
  ✅ edos                                 20,000 rows  |  {'safe': 15146, 'harmful': 4854}

[5] CASH
  ✅ cash                                 18,096 rows  |  {'harmful': 18096}

[6] TweetEval Offensive
  ✅ tweeteval_offensive                  11,916 rows  |  {'safe': 7975, 'harmful': 3941}

[7] TweetEval Hate
  ✅ tweeteval_hate                        8,987 rows  |  {'safe': 5206, 'harmful': 3781}

[8] MLMA
  ✅ mlma                                 18,653 rows  |  {'harmful': 15601, 'safe': 3052}

[9] UCB MHS (pre-filtered)
  ✅ ucb_mhs                               2,120 rows  |  {'harmful': 2120}

[10] Kaggle Cyberbullying
  ✅ kaggle_cy

In [15]:
# ============================================================
# CELL 4 — Inject PersonaChat + DailyDialog + Synthetic Hard Negatives  [V4]
# V4: Hard negatives expanded 2K → 5K with teen slang patterns
# ============================================================

print("="*70)
print("CELL 4: SAFE INJECTION + FILTER + DEDUP + BALANCE")
print("="*70)

# ── STEP 1: Load DailyDialog ──────────────────────────────────
print("\n📌 STEP 1: Loading DailyDialog...")
dd_path = RAW / "dailyDialog"
dd_utterances = []
if dd_path.exists():
    for split_file in dd_path.glob("**/*.txt"):
        with open(split_file) as f:
            for line in f:
                for utt in line.strip().split("__eou__"):
                    utt = utt.strip()
                    if utt:
                        dd_utterances.append(utt)
elif (RAW / "dailydialog").exists():
    dd_path = RAW / "dailydialog"
    for split_file in dd_path.glob("**/*.txt"):
        with open(split_file) as f:
            for line in f:
                for utt in line.strip().split("__eou__"):
                    utt = utt.strip()
                    if utt:
                        dd_utterances.append(utt)

print(f"  Raw utterances : {len(dd_utterances):,}")

# ── STEP 2: Load PersonaChat ──────────────────────────────────
print("\nLoading PersonaChat...")
pc_utterances = []
pc_path = RAW / "ChatPersona" / "personaChat"

if pc_path.exists():
    for split_file in pc_path.glob("*.parquet"):  # Changed to look for .parquet
        df_pc_raw = pd.read_parquet(split_file)

        # Iterate through columns to find text/lists of text (history, candidates, etc)
        for col in df_pc_raw.columns:
            for item in df_pc_raw[col].dropna():
                if isinstance(item, (list, np.ndarray)):
                    # Extract strings from lists (like chat histories)
                    pc_utterances.extend([str(x) for x in item if isinstance(x, str)])
                elif isinstance(item, str):
                    # Extract direct strings
                    pc_utterances.append(item)

print(f"  Raw utterances : {len(pc_utterances):,}")


# ── STEP 3: Normalize + deduplicate dialogue sources ──────────
print("\n📌 STEP 3: Normalize + dedup dialogue sources")

# Normalize and deduplicate
dd_norm = list(set(filter(None, [normalize_text(u) for u in dd_utterances])))
pc_norm = list(set(filter(None, [normalize_text(u) for u in pc_utterances])))

# Filter: >= 3 words
dd_norm = [u for u in dd_norm if len(u.split()) >= 3]
pc_norm = [u for u in pc_norm if len(u.split()) >= 3]

print(f"  DailyDialog after norm+dedup+filter: {len(dd_norm):,}")
print(f"  PersonaChat after norm+dedup+filter: {len(pc_norm):,}")

# Cap each at 20K
np.random.seed(42)
if len(dd_norm) > 20_000:
    dd_norm = list(np.random.choice(dd_norm, 20_000, replace=False))
    print(f"  DailyDialog capped at 20,000")
if len(pc_norm) > 20_000:
    pc_norm = list(np.random.choice(pc_norm, 20_000, replace=False))
    print(f"  PersonaChat capped at 20,000")

# Add to df_all
df_dd = pd.DataFrame({'text': dd_norm, 'm1_label': 'safe', 'source_dataset': 'daily_dialog'})
df_pc = pd.DataFrame({'text': pc_norm, 'm1_label': 'safe', 'source_dataset': 'personachat'})
df_all = pd.concat([df_all, df_dd, df_pc], ignore_index=True)
print(f"\ndf_all after dialogue injection: {len(df_all):,}")

# ── STEP 4: Generate Synthetic Hard Negatives ─────────────────
print("\n📌 STEP 4: Generate synthetic hard negatives")
import itertools

# V4: expanded from 26 → 50+ templates with teen slang patterns
safe_templates = [
    # ── Original safe patterns ──
    "i like {}", "let's {} tomorrow", "wanna {} later?",
    "come {} with me", "let's meet at {}", "did you try {}",
    "this {} will make you laugh", "that {} is killer",
    "{} is deadly good", "i'm dying to see {}", "this {} slaps",
    "you should check out {}", "i love that {}",
    "have you seen {}?", "{} is my favorite",
    "can you recommend a {}", "what {} do you like",
    "i just watched a great {}", "that {} was amazing",
    "let's go get some {}", "i need a new {}",
    "{} is so good", "the {} was incredible",
    "i can't stop thinking about that {}",
    "you gotta try this {}", "this {} hits different",
    # ── V4: Teen slang / Gen-Z patterns (high FP risk) ──
    "no cap that {} is fire", "lowkey obsessed with this {}",
    "that {} is bussin fr fr", "slay queen energy at the {}",
    "this {} is giving main character vibes", "the {} ate and left no crumbs",
    "that {} was lowkey mid tho", "deadass the best {} ever",
    "ngl this {} goes hard", "caught a {} and it was elite",
    "im dead this {} is hilarious", "that {} had me screaming",
    "bro this {} is goated", "the {} was valid honestly",
    "finna {} after school", "boutta hit up the {}",
    "why is this {} so fire", "rent free in my head this {}",
    "you're sleeping on this {}", "this {} is criminally underrated",
    # ── V4: WhatsApp-style safe chat ──
    "wanna come over and {} tonight", "my mom said we can {} this weekend",
    "lmk when you wanna {}", "bet let's {} after class",
    "yo send me that {} link", "can you bring the {} tmrw",
    "i'll be at the {} around 5", "meet me at the {} after school",
]
# V4: expanded fillers for more coverage
fillers = ["game", "movie", "song", "show", "restaurant", "park",
           "cafe", "class", "gym", "place", "series", "album",
           "book", "recipe", "playlist", "podcast", "video",
           "drink", "spot", "track", "tiktok", "reel", "stream",
           "concert", "match", "level", "episode", "vlog", "meme",
           "hangout", "party"]

synthetic = []
for tmpl, fill in itertools.product(safe_templates, fillers):
    synthetic.append(normalize_text(tmpl.format(fill)))

# Deduplicate and cap
synthetic = list(set(filter(None, synthetic)))
np.random.seed(42)
if len(synthetic) > 5000:  # V4: raised from 2K → 5K
    synthetic = list(np.random.choice(synthetic, 5000, replace=False))

df_syn = pd.DataFrame({'text': synthetic, 'm1_label': 'safe', 'source_dataset': 'synthetic_safe'})
df_all = pd.concat([df_all, df_syn], ignore_index=True)
print(f"  Generated {len(synthetic)} synthetic safe examples")
print(f"  df_all after synthetic injection: {len(df_all):,}")

# ── STEP 5: Filter (>= 3 words, has letters) ──────────────────
print("\n📌 STEP 5: Filter")
before = len(df_all)
df_all = df_all[df_all['text'].notna() & (df_all['text'].str.len() > 0)]
df_all['word_count'] = df_all['text'].str.split().str.len()
df_all = df_all[df_all['word_count'] >= 3].copy()
df_all = df_all[df_all['text'].str.contains(r'[a-zA-Z]', regex=True)].copy()
print(f"  Dropped: {before - len(df_all):,} rows")
print(f"  Remaining: {len(df_all):,} rows")

# ── STEP 6: Exact dedup ───────────────────────────────────────
print("\n📌 STEP 6: Exact dedup")
before = len(df_all)
df_all['text_hash'] = df_all['text'].apply(text_hash)
df_all = df_all.drop_duplicates(subset='text_hash', keep='first').copy()
df_all = df_all.drop(columns=['text_hash', 'word_count'])
print(f"  Duplicates removed: {before - len(df_all):,}")
print(f"  After dedup: {len(df_all):,}")

# ── STEP 7: Show safe source diversity ────────────────────────
print("\n📌 STEP 7: Safe source diversity")
safe_sources = df_all[df_all['m1_label'] == 'safe']['source_dataset'].value_counts()
print(f"\n  Safe sources now:")
for src, cnt in safe_sources.items():
    print(f"    {src:<25} {cnt:>8,}")
print(f"\n  Total safe   : {(df_all['m1_label']=='safe').sum():,}")
print(f"  Total harmful: {(df_all['m1_label']=='harmful').sum():,}")

print(f"\n{'='*70}")
print("✅ CELL 4 COMPLETE — All sources loaded, normalized, deduped")
print("   Next: Cell 5 (balanced sampling to 50K/50K)")
print(f"{'='*70}")

CELL 4: SAFE INJECTION + FILTER + DEDUP + BALANCE

📌 STEP 1: Loading DailyDialog...
  Raw utterances : 129,215

Loading PersonaChat...
  Raw utterances : 84,848

📌 STEP 3: Normalize + dedup dialogue sources
  DailyDialog after norm+dedup+filter: 89,906
  PersonaChat after norm+dedup+filter: 6,732
  DailyDialog capped at 20,000

df_all after dialogue injection: 439,324

📌 STEP 4: Generate synthetic hard negatives
  Generated 1674 synthetic safe examples
  df_all after synthetic injection: 440,998

📌 STEP 5: Filter
  Dropped: 10,337 rows
  Remaining: 430,661 rows

📌 STEP 6: Exact dedup
  Duplicates removed: 15,413
  After dedup: 415,248

📌 STEP 7: Safe source diversity

  Safe sources now:
    jigsaw                     142,935
    gab                         22,967
    daily_dialog                18,035
    edos                        15,058
    stormfront                   8,997
    hatexplain                   8,099
    tweeteval_offensive          7,683
    kaggle_cyberbullying      

In [16]:
# ============================================================
# CELL 5 — Balanced Sampling: 60K Safe / 60K Harmful  [V4]
# ============================================================

print("="*70)
print("CELL 5: BALANCED SAMPLING — 60K/60K  [V4]")
print("="*70)

TARGET_PER_CLASS = 60_000   # V4: raised from 50K → 60K

def sample_source(df, source, label, n, note=''):
    """Sample n rows from a specific source+label. n=0 means take all."""
    sub = df[(df['source_dataset']==source) & (df['m1_label']==label)]
    taken = sub.sample(n=min(n, len(sub)), random_state=42) if n else sub
    print(f"  {source:<25} {label:<8}  available={len(sub):>7,}  taken={len(taken):>7,}  {note}")
    return taken

# ══════════════════════════════════════════════════════════════
# SAFE POOL — 11 sources for maximum register diversity
# ══════════════════════════════════════════════════════════════
print("\n  --- SAFE POOL (11 sources) ---")
safe_frames = []

# Core dialogue (base coverage)
safe_frames.append(sample_source(df_all, 'personachat',         'safe', 20_000, 'dialogue — V4 raised'))  # V4: 15K→20K
safe_frames.append(sample_source(df_all, 'daily_dialog',        'safe', 20_000, 'dialogue — V4 raised'))  # V4: 15K→20K

# Social media hard negatives (same register as harmful)
safe_frames.append(sample_source(df_all, 'kaggle_cyberbullying','safe',  5_000, 'tweets — hard neg'))
safe_frames.append(sample_source(df_all, 'tweeteval_offensive', 'safe',  4_000, 'tweets — hard neg'))
safe_frames.append(sample_source(df_all, 'davidson',           'safe',  3_000, 'tweets — hard neg'))
safe_frames.append(sample_source(df_all, 'edos',               'safe',  3_000, 'gender-topic hard neg'))
safe_frames.append(sample_source(df_all, 'olid',               'safe',  2_000, 'tweets — hard neg'))
safe_frames.append(sample_source(df_all, 'stormfront',         'safe',  1_500, 'forum — hardest neg'))
safe_frames.append(sample_source(df_all, 'hasoc',              'safe',  1_500, 'EN tweets — hard neg'))

# LLM / diverse coverage
safe_frames.append(sample_source(df_all, 'aegis_safety',       'safe',      0, 'LLM-chat'))

# Synthetic hard negatives
safe_frames.append(sample_source(df_all, 'synthetic_safe',     'safe',      0, 'synthetic patterns'))

df_safe = pd.concat(safe_frames, ignore_index=True)
df_safe = df_safe.drop_duplicates(subset='text')

# Cap to target
if len(df_safe) > TARGET_PER_CLASS:
    df_safe = df_safe.sample(n=TARGET_PER_CLASS, random_state=42)
    print(f"\n  Safe pool capped to {TARGET_PER_CLASS:,}")

print(f"\n  SAFE TOTAL: {len(df_safe):,}")

# ══════════════════════════════════════════════════════════════
# HARMFUL POOL — Same tiered approach as V2
# ══════════════════════════════════════════════════════════════
print("\n  --- HARMFUL POOL ---")
harmful_frames = []

# Tier 1: Take ALL (rare/valuable sources)
harmful_frames.append(sample_source(df_all, 'aegis_safety',    'harmful', 0, 'chat domain'))
harmful_frames.append(sample_source(df_all, 'convabuse',       'harmful', 0, 'chat domain'))
harmful_frames.append(sample_source(df_all, 'implicit_hate',   'harmful', 0, 'demographic hate'))
harmful_frames.append(sample_source(df_all, 'hammer',          'harmful', 0, 'threats'))
harmful_frames.append(sample_source(df_all, 'ethos',           'harmful', 0, 'threats'))
harmful_frames.append(sample_source(df_all, 'cash',            'harmful', 0, 'sexual harassment'))
harmful_frames.append(sample_source(df_all, 'edos',            'harmful', 0, 'sexism'))
harmful_frames.append(sample_source(df_all, 'ucb_mhs',        'harmful', 0, 'violence'))
harmful_frames.append(sample_source(df_all, 'stormfront',      'harmful', 0, 'white supremacy'))
harmful_frames.append(sample_source(df_all, 'hasoc',           'harmful', 0, 'EN tweets hate'))

# Tier 2: Capped (large noisy sources)
harmful_frames.append(sample_source(df_all, 'hatexplain',      'harmful', 10_000, 'hate+offensive — V4'))
harmful_frames.append(sample_source(df_all, 'davidson',        'harmful',  8_000, 'hate+offensive — V4'))
harmful_frames.append(sample_source(df_all, 'toxigen',         'harmful',  6_000, 'adversarial — V4'))
harmful_frames.append(sample_source(df_all, 'mlma',            'harmful',  4_000, 'multilabel'))
harmful_frames.append(sample_source(df_all, 'kaggle_cyberbullying', 'harmful', 3_000, 'cyberbullying'))
harmful_frames.append(sample_source(df_all, 'tweeteval_offensive', 'harmful', 2_000, 'offensive tweets'))
harmful_frames.append(sample_source(df_all, 'tweeteval_hate',   'harmful', 2_000, 'hate tweets'))
harmful_frames.append(sample_source(df_all, 'olid',            'harmful', 2_000, 'offensive'))
harmful_frames.append(sample_source(df_all, 'gab',             'harmful', 2_000, 'social media'))

df_harmful = pd.concat(harmful_frames, ignore_index=True)
df_harmful = df_harmful.drop_duplicates(subset='text')

# Cap to target
if len(df_harmful) > TARGET_PER_CLASS:
    df_harmful = df_harmful.sample(n=TARGET_PER_CLASS, random_state=42)
    print(f"\n  Harmful pool capped to {TARGET_PER_CLASS:,}")

print(f"\n  HARMFUL TOTAL: {len(df_harmful):,}")

# ══════════════════════════════════════════════════════════════
# COMBINE + FORMAT
# ══════════════════════════════════════════════════════════════
print("\n📌 Combining safe + harmful...")
df_m1 = pd.concat([df_safe, df_harmful], ignore_index=True)
df_m1 = df_m1.sample(frac=1, random_state=42).reset_index(drop=True)

df_m1['language']       = 'en'
df_m1['routing_method'] = 'manual'
df_m1['ambiguous']      = False
df_m1['confidence']     = None
df_m1 = df_m1.rename(columns={'m1_label': 'label'})
df_m1 = df_m1[['text','label','language','source_dataset','routing_method','ambiguous','confidence']]

print("\n" + "="*70)
print("M1 V4 CORPUS — FINAL SUMMARY")
print("="*70)
print(f"Total rows   : {len(df_m1):,}")
print(f"\nLabel split  :")
print(df_m1['label'].value_counts().to_string())
print(f"\nPer source   :")
print(df_m1.groupby(['source_dataset','label']).size().unstack(fill_value=0).to_string())

# Quick validation
safe_pct = (df_m1['label']=='safe').mean() * 100
print(f"\n📊 Balance: {safe_pct:.1f}% safe / {100-safe_pct:.1f}% harmful")
n_sources = df_m1['source_dataset'].nunique()
print(f"📊 Source diversity: {n_sources} unique sources")

print(f"\n{'='*70}")
print("✅ CELL 5 COMPLETE — Balanced M1 V4 corpus ready (60K/60K)")
print("   Next: Cell 6 (stratified split + save)")
print(f"{'='*70}")

CELL 5: BALANCED SAMPLING — 60K/60K  [V4]

  --- SAFE POOL (11 sources) ---
  personachat               safe      available=  6,727  taken=  6,727  dialogue — V4 raised
  daily_dialog              safe      available= 18,035  taken= 18,035  dialogue — V4 raised
  kaggle_cyberbullying      safe      available=  7,436  taken=  5,000  tweets — hard neg
  tweeteval_offensive       safe      available=  7,683  taken=  4,000  tweets — hard neg
  davidson                  safe      available=  4,021  taken=  3,000  tweets — hard neg
  edos                      safe      available= 15,058  taken=  3,000  gender-topic hard neg
  olid                      safe      available=  2,570  taken=  2,000  tweets — hard neg
  stormfront                safe      available=  8,997  taken=  1,500  forum — hardest neg
  hasoc                     safe      available=  6,901  taken=  1,500  EN tweets — hard neg
  aegis_safety              safe      available=    995  taken=    995  LLM-chat
  synthetic_safe  

In [17]:
# ============================================================
# CELL 5b — Golden Dataset Injection for M1  [V4 NEW]
# ============================================================
# Same golden_dataset_EN.json used in M2, but here we inject
# ALL entries (safe + harmful) as binary labels for M1.
# Strategy:
#   • Normal golden → 5x oversample
#   • ml_corrected=True → 10x (hardest edge cases)
# ============================================================

import json, warnings

GOLDEN_JSON = Path('/content/drive/MyDrive/AEGIS_ML/data/golden_dataset_EN.json')
OVERSAMPLE_NORMAL   = 5
OVERSAMPLE_ML_WRONG = 10

try:
    with open(GOLDEN_JSON) as f:
        golden_raw = json.load(f)
    print(f'Loaded {len(golden_raw)} golden entries')

    golden_rows = []
    for entry in golden_raw:
        raw_label = entry.get('human_label') or entry.get('primary_class', '')

        # Binary mapping for M1
        if raw_label in ('safe', 'uncertain'):
            m1_label = 'safe'
        elif raw_label in ('threat', 'verbal_harassment', 'sexual_harassment',
                           'SEXUAL_HARASSMENT', 'discrimination', 'grooming'):
            m1_label = 'harmful'
        else:
            continue

        text = entry.get('normalized_text') or entry.get('raw_text', '')
        text = normalize_text(str(text))
        if len(text.split()) < 3:
            continue

        is_ml_corrected = bool(entry.get('ml_corrected', False))
        n_copies = OVERSAMPLE_ML_WRONG if is_ml_corrected else OVERSAMPLE_NORMAL

        for _ in range(n_copies):
            golden_rows.append({
                'text': text,
                'label': m1_label,
                'source_dataset': 'golden_production',
                'language': 'en',
                'routing_method': 'golden',
                'ambiguous': False,
                'confidence': None,
            })

    df_golden = pd.DataFrame(golden_rows).drop_duplicates(subset='text')
    print(f'Golden M1 entries: {len(df_golden)} rows (after dedup)')
    print(df_golden['label'].value_counts().to_string())

    # Merge: golden takes priority in dedup
    df_combined = pd.concat([df_golden, df_m1], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset='text', keep='first')
    df_m1 = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f'\n✅ After golden injection: {len(df_m1):,} total rows')
    print(df_m1['label'].value_counts().to_string())

except FileNotFoundError:
    warnings.warn(f'⚠️  golden_dataset_EN.json not found at {GOLDEN_JSON} — skipping')
    print('Continuing without golden dataset.')


Loaded 300 golden entries
Golden M1 entries: 255 rows (after dedup)
label
harmful    135
safe       120

✅ After golden injection: 107,681 total rows
label
harmful    60131
safe       47550


In [18]:
# ============================================================
# CELL 6 — Stratified 70/15/15 Split + Save
# ============================================================

from sklearn.model_selection import train_test_split

print("="*70)
print("CELL 6: STRATIFIED SPLIT + SAVE")
print("="*70)

# ── STEP 1: Stratified split ──────────────────────────────────
print("\n📌 STEP 1: Stratified 70/15/15 split")

train, temp = train_test_split(
    df_m1, test_size=0.30, random_state=42,
    stratify=df_m1['label']
)
val, test = train_test_split(
    temp, test_size=0.50, random_state=42,
    stratify=temp['label']
)

print(f"  Train: {len(train):,} rows ({len(train)/len(df_m1)*100:.1f}%)")
print(f"  Val:   {len(val):,} rows ({len(val)/len(df_m1)*100:.1f}%)")
print(f"  Test:  {len(test):,} rows ({len(test)/len(df_m1)*100:.1f}%)")
print(f"  Total: {len(train)+len(val)+len(test):,} rows")

# ── STEP 2: Verify label balance per split ────────────────────
print("\n📌 STEP 2: Label balance per split")
for name, split_df in [('Train', train), ('Val', val), ('Test', test)]:
    counts = split_df['label'].value_counts()
    pct_safe = (split_df['label']=='safe').mean()*100
    print(f"  {name}: safe={counts.get('safe',0):,} harmful={counts.get('harmful',0):,} ({pct_safe:.1f}%/{100-pct_safe:.1f}%)")

# ── STEP 3: Cross-split leakage check ────────────────────────
print("\n📌 STEP 3: Cross-split leakage check")
train_texts = set(train['text'])
val_texts   = set(val['text'])
test_texts  = set(test['text'])

tv_leak = len(train_texts & val_texts)
tt_leak = len(train_texts & test_texts)
vt_leak = len(val_texts & test_texts)
print(f"  Train∩Val  overlap: {tv_leak} (should be 0)")
print(f"  Train∩Test overlap: {tt_leak} (should be 0)")
print(f"  Val∩Test   overlap: {vt_leak} (should be 0)")

if tv_leak + tt_leak + vt_leak == 0:
    print("  ✅ No leakage detected")
else:
    print("  ⚠️  WARNING: Leakage detected!")

# ── STEP 4: Source distribution per split ────────────────────
print("\n📌 STEP 4: Source distribution per split")
for name, split_df in [('Train', train), ('Val', val), ('Test', test)]:
    print(f"\n  {name}:")
    src_counts = split_df['source_dataset'].value_counts()
    for src, cnt in src_counts.head(15).items():
        pct = cnt / len(split_df) * 100
        print(f"    {src:<25} {cnt:>6,} ({pct:>5.1f}%)")

# ── STEP 5: Normalization validation ──────────────────────────
print("\n📌 STEP 5: Normalization validation")
has_upper = df_m1['text'].str.contains(r'[A-Z]', regex=True).sum()
has_url   = df_m1['text'].str.contains(r'https?://', regex=True).sum()
print(f"  Texts with uppercase: {has_upper} (should be 0)")
print(f"  Texts with URLs: {has_url} (should be 0)")

# ── STEP 6: Save ─────────────────────────────────────────────
print("\n📌 STEP 6: Save to Drive")
OUT = PROCESSED

train.to_csv(OUT / "m1_trainV4.csv", index=False)
val.to_csv(OUT / "m1_valV4.csv", index=False)
test.to_csv(OUT / "m1_testV4.csv", index=False)

for name, fpath in [('train', OUT/'m1_trainV4.csv'),
                     ('val',   OUT/'m1_valV4.csv'),
                     ('test',  OUT/'m1_testV4.csv')]:
    size_kb = fpath.stat().st_size / 1024
    print(f"  ✅ {fpath.name:<25} {size_kb:>8.1f} KB")

# ── STEP 7: Final spot check ─────────────────────────────────
print("\n📌 STEP 7: Random sample (10 safe + 10 harmful)")
print("\n  SAFE EXAMPLES:")
for _, row in train[train['label']=='safe'].sample(10, random_state=42).iterrows():
    print(f"    [{row['source_dataset']:<20}] {row['text'][:80]}")

print("\n  HARMFUL EXAMPLES:")
for _, row in train[train['label']=='harmful'].sample(10, random_state=42).iterrows():
    print(f"    [{row['source_dataset']:<20}] {row['text'][:80]}")

print("\n" + "="*70)
print("🎉 M1 V4 CORPUS BUILD COMPLETE")
print("="*70)
print(f"\n  📊 Total: {len(df_m1):,} rows")
print(f"  📊 Balance: {(df_m1['label']=='safe').mean()*100:.1f}% safe / {(df_m1['label']=='harmful').mean()*100:.1f}% harmful")
print(f"  📊 Sources: {df_m1['source_dataset'].nunique()} unique")
print(f"  📊 Splits: train={len(train):,} / val={len(val):,} / test={len(test):,}")
print(f"  📊 Normalization: ALL text lowercase, no URLs, no @mentions")
print(f"\n  📁 Files saved to: {OUT}")
print(f"     m1_trainV4.csv / m1_valV4.csv / m1_testV4.csv")
print(f"{'='*70}")

CELL 6: STRATIFIED SPLIT + SAVE

📌 STEP 1: Stratified 70/15/15 split
  Train: 75,376 rows (70.0%)
  Val:   16,152 rows (15.0%)
  Test:  16,153 rows (15.0%)
  Total: 107,681 rows

📌 STEP 2: Label balance per split
  Train: safe=33,285 harmful=42,091 (44.2%/55.8%)
  Val: safe=7,132 harmful=9,020 (44.2%/55.8%)
  Test: safe=7,133 harmful=9,020 (44.2%/55.8%)

📌 STEP 3: Cross-split leakage check
  Train∩Val  overlap: 0 (should be 0)
  Train∩Test overlap: 0 (should be 0)
  Val∩Test   overlap: 0 (should be 0)
  ✅ No leakage detected

📌 STEP 4: Source distribution per split

  Train:
    daily_dialog              12,560 ( 16.7%)
    cash                       9,384 ( 12.4%)
    davidson                   6,664 (  8.8%)
    hatexplain                 5,869 (  7.8%)
    kaggle_cyberbullying       5,212 (  6.9%)
    edos                       4,863 (  6.5%)
    personachat                4,693 (  6.2%)
    hasoc                      4,001 (  5.3%)
    tweeteval_offensive        3,964 (  5.3%)
    